In [55]:
# Filename: local_dev.py
from starlette.requests import Request

from ray import serve
from ray.serve.handle import DeploymentHandle, DeploymentResponse


@serve.deployment
class Doubler:
    def double(self, s: str):
        return s + " " + s


@serve.deployment
class HelloDeployment:
    def __init__(self, doubler: DeploymentHandle):
        self.doubler = doubler

    async def say_hello_twice(self, name: str):
        return await self.doubler.double.remote(f"Hello, {name}!")

    async def __call__(self, request: Request):
        return await self.say_hello_twice(request.query_params["name"])


app = HelloDeployment.bind(Doubler.bind())

In [56]:
handle = serve.run(app)

INFO 2026-05-29 17:15:10,571 serve 2032472 -- Connecting to existing Serve app in namespace "serve". New http options will not be applied.
(ServeController pid=2033966) INFO 2026-05-29 17:15:10,631 controller 2033966 -- Deploying new version of Deployment(name='Doubler', app='default') (initial target replicas: 1).
(ServeController pid=2033966) INFO 2026-05-29 17:15:10,632 controller 2033966 -- Deploying new version of Deployment(name='HelloDeployment', app='default') (initial target replicas: 1).
(ServeController pid=2033966) INFO 2026-05-29 17:15:10,747 controller 2033966 -- Stopping 1 replicas of Deployment(name='Doubler', app='default') with outdated versions.
(ServeController pid=2033966) INFO 2026-05-29 17:15:10,747 controller 2033966 -- Adding 1 replica to Deployment(name='Doubler', app='default').
(ServeController pid=2033966) INFO 2026-05-29 17:15:10,751 controller 2033966 -- Stopping 1 replicas of Deployment(name='HelloDeployment', app='default') with outdated versions.
(Serv

In [57]:
for _ in range(1_000):
    ref = handle.say_hello_twice.remote("yso")
    await ref

(ProxyActor pid=2635038, ip=10.0.1.10) INFO 2026-05-30 00:08:39,768 proxy 10.0.1.10 -- Got updated endpoints: {Deployment(name='HelloDeployment', app='default'): EndpointInfo(route='/', app_is_cross_language=False, route_patterns=None)}.
(ProxyActor pid=2635038, ip=10.0.1.10) INFO 2026-05-30 00:08:39,781 proxy 10.0.1.10 -- Started <ray.serve._private.router.SharedRouterLongPollClient object at 0x79c0d85552b0>.
(ServeReplica:default:Doubler pid=2036187) /home/nico/miniconda3/envs/drp/lib/python3.14/site-packages/ray/serve/_private/replica.py:3140: UserWarning: Calling sync method 'double' directly on the asyncio loop. In a future version, sync methods will be run in a threadpool by default. Ensure your sync methods are thread safe or keep the existing behavior by making them `async def`. Opt into the new behavior by setting RAY_SERVE_RUN_SYNC_IN_THREADPOOL=1.
(ServeReplica:default:Doubler pid=2036187)   warnings.warn(
(ServeReplica:default:Doubler pid=2036187) INFO 2026-05-29 17:15:14,6

In [51]:
await ref

'Hello, yso! Hello, yso!'